In [1]:
!pip -q install pyspark
import sys, pyspark
print(sys.version)
print(pyspark.__version__)

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
4.0.4


In [2]:
from sklearn.datasets import fetch_covtype
import pandas as pd

df = fetch_covtype(as_frame=True).frame.copy()
df["label"] = (df["Cover_Type"] == 1).astype(int)
df = df.drop(columns="Cover_Type")
print(df.shape)
print(df["label"].value_counts(normalize=True))
df.to_csv("/content/covtype.csv", index=False)

(581012, 55)
label
0    0.635395
1    0.364605
Name: proportion, dtype: float64


In [3]:
import time
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier, LinearSVC, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = (SparkSession.builder.master("local[*]")
         .appName("mllib-replication").config("spark.driver.memory", "4g").getOrCreate())

def load_spark(path):
    sdf = spark.read.csv(path, header=True, inferSchema=True)
    feats = [c for c in sdf.columns if c != "label"]
    sdf = (VectorAssembler(inputCols=feats, outputCol="features")
           .transform(sdf).select("features", "label"))
    train, test = sdf.randomSplit([0.7, 0.3], seed=42)
    train, test = train.cache(), test.cache()
    print("train:", train.count(), "test:", test.count())
    return train, test

def run_spark(dataset, path, runs=3):
    train, test = load_spark(path)
    models = {
        "DecisionTree": DecisionTreeClassifier(maxBins=32, maxDepth=6, minInfoGain=0.15, impurity="entropy"),
        "SVM": LinearSVC(maxIter=1000),
        "LogReg": LogisticRegression(maxIter=1000),
    }
    ev = BinaryClassificationEvaluator(metricName="areaUnderROC")
    rows = []
    for name, m in models.items():
        times = []
        for _ in range(runs):
            t0 = time.time(); fitted = m.fit(train); times.append(time.time() - t0)
        auc = ev.evaluate(fitted.transform(test))
        rows.append({"dataset": dataset, "tool": "Spark MLlib", "model": name,
                     "train_time_s": round(sum(times)/runs, 2), "auc": round(auc, 4)})
        print(rows[-1])
    return rows

results = run_spark("Covtype", "/content/covtype.csv")
pd.DataFrame(results)

train: 406550 test: 174462
{'dataset': 'Covtype', 'tool': 'Spark MLlib', 'model': 'DecisionTree', 'train_time_s': 9.6, 'auc': 0.7471}
{'dataset': 'Covtype', 'tool': 'Spark MLlib', 'model': 'SVM', 'train_time_s': 62.79, 'auc': 0.8406}
{'dataset': 'Covtype', 'tool': 'Spark MLlib', 'model': 'LogReg', 'train_time_s': 12.46, 'auc': 0.8428}


,dataset,tool,model,train_time_s,auc
0,Covtype,Spark MLlib,DecisionTree,9.60,0.7471
1,Covtype,Spark MLlib,SVM,62.79,0.8406
2,Covtype,Spark MLlib,LogReg,12.46,0.8428


In [4]:
import time, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier as SkDT
from sklearn.svm import LinearSVC as SkSVM
from sklearn.linear_model import LogisticRegression as SkLR
from sklearn.metrics import roc_auc_score

def run_sklearn(dataset, path, runs=3):
    d = pd.read_csv(path)
    X, y = d.drop(columns="label").values, d["label"].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
    models = {
        "DecisionTree": SkDT(criterion="entropy", max_depth=6, min_impurity_decrease=0.15, random_state=42),
        "SVM": make_pipeline(StandardScaler(), SkSVM(max_iter=1000, dual=False)),
        "LogReg": make_pipeline(StandardScaler(), SkLR(max_iter=1000)),
    }
    rows = []
    for name, m in models.items():
        times = []
        for _ in range(runs):
            t0 = time.time(); m.fit(Xtr, ytr); times.append(time.time() - t0)
        score = m.predict_proba(Xte)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(Xte)
        rows.append({"dataset": dataset, "tool": "scikit-learn", "model": name,
                     "train_time_s": round(sum(times)/runs, 2), "auc": round(roc_auc_score(yte, score), 4)})
        print(rows[-1])
    return rows

results += run_sklearn("Covtype", "/content/covtype.csv")
res = pd.DataFrame(results)
res.to_csv("/content/hasil_covtype.csv", index=False)
res

{'dataset': 'Covtype', 'tool': 'scikit-learn', 'model': 'DecisionTree', 'train_time_s': 1.29, 'auc': np.float64(0.7464)}
{'dataset': 'Covtype', 'tool': 'scikit-learn', 'model': 'SVM', 'train_time_s': 24.05, 'auc': np.float64(0.8425)}
{'dataset': 'Covtype', 'tool': 'scikit-learn', 'model': 'LogReg', 'train_time_s': 4.61, 'auc': np.float64(0.843)}


,dataset,tool,model,train_time_s,auc
0,Covtype,Spark MLlib,DecisionTree,9.60,0.7471
1,Covtype,Spark MLlib,SVM,62.79,0.8406
2,Covtype,Spark MLlib,LogReg,12.46,0.8428
3,Covtype,scikit-learn,DecisionTree,1.29,0.7464
4,Covtype,scikit-learn,SVM,24.05,0.8425
5,Covtype,scikit-learn,LogReg,4.61,0.8430
